# Chapter 20 — What It Means for a Prediction to Be Good

Earlier models produced probability distributions and used them to sample generated tokens.

This chapter asks the basic evaluation question behind those distributions: how much probability did the model assign to the token that actually came next?

By the end of this chapter, you should be able to:

- identify an input context and its correct next-token target;
- validate a token probability distribution;
- extract the probability assigned to the target;
- compare two predictions for one observed example;
- distinguish the top predicted token from the correct token;
- explain why sampling and evaluation answer different questions;
- explain why probability one is not required for good language modeling; and
- connect target probability to evaluation over many examples and later loss functions.

This chapter evaluates fake predictions without training a model.


## The Correct Next Token

During evaluation, the **correct next token** is the token that actually follows the context in held-out text.

For the context `"the cat sat on the"`, suppose the observed continuation is `"mat"`.

The model predicts a probability distribution before being shown that target.

Evaluation then reads the probability stored at the target token `"mat"`.


## One-Example Comparison

Suppose Model A assigns `"mat"` probability `0.60`, while Model B assigns it probability `0.01`.

Model A is better for this observed example because it assigned more probability to what happened.

The rest of each distribution still matters for other possible examples, but one target supplies one direct lookup for this example.


## Generation and Evaluation Ask Different Questions

Generation asks which token to choose from the model's distribution.

Evaluation already knows the target and asks how much probability that target received.

A sampled token is one random outcome, while the prediction is the full distribution from which that outcome could be sampled.


## Use a Tiny Readable Vocabulary

Word-like tokens keep the probability examples easy to read, although the same logic applies to characters and subwords.


In [1]:
vocabulary = [
    "mat",
    "rug",
    "yard",
    "food",
    "dog",
    "cat",
    "bird",
    "tree",
    ".",
]

print("Vocabulary:")
for token_id, token in enumerate(vocabulary):
    print(f"{token_id:>2}: {token}")

Vocabulary:
 0: mat
 1: rug
 2: yard
 3: food
 4: dog
 5: cat
 6: bird
 7: tree
 8: .


## Validate a Prediction Distribution

A valid prediction contains exactly the vocabulary tokens, has no negative values, and sums to one within a floating-point tolerance.


In [2]:
def assert_valid_prediction(
    prediction: dict[str, float],
    vocabulary: list[str],
) -> None:
    if not prediction:
        raise ValueError("A prediction cannot be empty.")

    expected_tokens = set(vocabulary)
    predicted_tokens = set(prediction)
    missing_tokens = expected_tokens - predicted_tokens
    extra_tokens = predicted_tokens - expected_tokens
    if missing_tokens or extra_tokens:
        raise ValueError(
            f"Prediction keys do not match the vocabulary. "
            f"Missing: {sorted(missing_tokens)}; extra: {sorted(extra_tokens)}."
        )

    for token, probability in prediction.items():
        if probability < 0:
            raise ValueError(
                f"Probability for token {token!r} is negative: {probability}."
            )

    probability_sum = sum(prediction.values())
    if abs(probability_sum - 1.0) > 1e-12:
        raise ValueError(
            f"Probabilities must sum to 1, but the sum is {probability_sum}."
        )

## Define the Main Evaluation Example

The context supplies the model input, and `"mat"` is the observed target from the text.


In [3]:
input_context = "the cat sat on the"
correct_next_token = "mat"

print("Input context:", repr(input_context))
print("Correct next token:", repr(correct_next_token))

Input context: 'the cat sat on the'
Correct next token: 'mat'


## Create Two Fake Predictions

Both models produce valid distributions over the same vocabulary, but they place very different probability on the target.


In [4]:
model_a_prediction = {
    "mat": 0.60,
    "rug": 0.25,
    "yard": 0.05,
    "food": 0.02,
    "dog": 0.02,
    "cat": 0.02,
    "bird": 0.01,
    "tree": 0.01,
    ".": 0.02,
}

model_b_prediction = {
    "mat": 0.01,
    "rug": 0.09,
    "yard": 0.30,
    "food": 0.20,
    "dog": 0.15,
    "cat": 0.10,
    "bird": 0.05,
    "tree": 0.05,
    ".": 0.05,
}

assert_valid_prediction(model_a_prediction, vocabulary)
assert_valid_prediction(model_b_prediction, vocabulary)
print("Both predictions are valid.")

Both predictions are valid.


## Extract the Target Probability

The central one-example quantity is the dictionary entry at the correct next token.


In [5]:
def target_probability(
    prediction: dict[str, float],
    correct_next_token: str,
    vocabulary: list[str],
) -> float:
    assert_valid_prediction(prediction, vocabulary)
    if correct_next_token not in prediction:
        raise ValueError(f"Correct token {correct_next_token!r} is not predicted.")
    return prediction[correct_next_token]


model_a_target_probability = target_probability(
    model_a_prediction,
    correct_next_token,
    vocabulary,
)
model_b_target_probability = target_probability(
    model_b_prediction,
    correct_next_token,
    vocabulary,
)

print("Model A target probability:", model_a_target_probability)
print("Model B target probability:", model_b_target_probability)

Model A target probability: 0.6
Model B target probability: 0.01


Model A is better on this example because `0.60` is greater than `0.01`.


## Print a Reusable Prediction Report

The report keeps context, target, target probability, and top prediction in one readable place.


In [6]:
def top_predicted_token(
    prediction: dict[str, float],
    vocabulary: list[str],
) -> tuple[str, float]:
    assert_valid_prediction(prediction, vocabulary)
    top_token = max(prediction, key=prediction.__getitem__)
    return top_token, prediction[top_token]


def print_prediction_report(
    model_name: str,
    input_context: str,
    correct_next_token: str,
    prediction: dict[str, float],
    vocabulary: list[str],
) -> None:
    assigned_probability = target_probability(
        prediction,
        correct_next_token,
        vocabulary,
    )
    top_token, top_probability = top_predicted_token(prediction, vocabulary)
    print("Model:", model_name)
    print("Input context:", repr(input_context))
    print("Correct next token:", repr(correct_next_token))
    print("Target probability:", assigned_probability)
    print("Top predicted token:", repr(top_token))
    print("Top probability:", top_probability)


print_prediction_report(
    "Model A",
    input_context,
    correct_next_token,
    model_a_prediction,
    vocabulary,
)
print()
print_prediction_report(
    "Model B",
    input_context,
    correct_next_token,
    model_b_prediction,
    vocabulary,
)

Model: Model A
Input context: 'the cat sat on the'
Correct next token: 'mat'
Target probability: 0.6
Top predicted token: 'mat'
Top probability: 0.6

Model: Model B
Input context: 'the cat sat on the'
Correct next token: 'mat'
Target probability: 0.01
Top predicted token: 'yard'
Top probability: 0.3


## Compare Two Predictions Programmatically

For one observed target, the higher target probability wins, and equal probabilities produce a tie.


In [7]:
def better_prediction_for_target(
    first_name: str,
    first_prediction: dict[str, float],
    second_name: str,
    second_prediction: dict[str, float],
    correct_next_token: str,
    vocabulary: list[str],
) -> str:
    first_probability = target_probability(
        first_prediction,
        correct_next_token,
        vocabulary,
    )
    second_probability = target_probability(
        second_prediction,
        correct_next_token,
        vocabulary,
    )
    if first_probability > second_probability:
        return first_name
    if second_probability > first_probability:
        return second_name
    return "tie"


better_model = better_prediction_for_target(
    "Model A",
    model_a_prediction,
    "Model B",
    model_b_prediction,
    correct_next_token,
    vocabulary,
)

print("Better prediction for this example:", better_model)
assert better_model == "Model A"

Better prediction for this example: Model A


## A Good Distribution Can Represent Alternatives

After `"the cat sat on the"`, both `"mat"` and `"rug"` are plausible continuations.

The observed target for this example remains `"mat"`, but assigning substantial probability to `"rug"` can represent genuine uncertainty.


In [8]:
reasonable_prediction = {
    "mat": 0.45,
    "rug": 0.35,
    "yard": 0.03,
    "food": 0.02,
    "dog": 0.03,
    "cat": 0.03,
    "bird": 0.02,
    "tree": 0.02,
    ".": 0.05,
}

assert_valid_prediction(reasonable_prediction, vocabulary)
print_prediction_report(
    "Reasonable uncertain model",
    input_context,
    correct_next_token,
    reasonable_prediction,
    vocabulary,
)

Model: Reasonable uncertain model
Input context: 'the cat sat on the'
Correct next token: 'mat'
Target probability: 0.45
Top predicted token: 'mat'
Top probability: 0.45


Its target probability `0.45` is lower than Model A's `0.60`, so Model A is still better for this one observed target under the chapter's comparison rule.

One example alone cannot determine whether either model's uncertainty is well calibrated across the language distribution.


## Probability One Looks Perfect on One Outcome

Assigning probability one to `"mat"` gives the highest possible target probability when `"mat"` is the observed target.

The same distribution assigns probability zero to every alternative and can fail maximally when another plausible token is observed.


In [9]:
overconfident_prediction = {
    "mat": 1.00,
    "rug": 0.00,
    "yard": 0.00,
    "food": 0.00,
    "dog": 0.00,
    "cat": 0.00,
    "bird": 0.00,
    "tree": 0.00,
    ".": 0.00,
}

assert_valid_prediction(overconfident_prediction, vocabulary)
print("Target probability when target is 'mat':")
print(target_probability(overconfident_prediction, "mat", vocabulary))
print("Target probability when target is 'rug':")
print(target_probability(overconfident_prediction, "rug", vocabulary))

Target probability when target is 'mat':
1.0
Target probability when target is 'rug':
0.0


The first result is optimal for that single outcome, while the second result exposes the risk of unjustified certainty.

Evaluation across many examples is required to determine whether confidence is consistently warranted.


## A Valid Distribution Can Still Be Bad

Mathematical validity only establishes that the values form a distribution.

Model B remains a poor prediction for this example because it assigns only `0.01` to the target.


In [10]:
assert_valid_prediction(model_b_prediction, vocabulary)
print("Model B is valid:", True)
print("Model B target probability:", model_b_prediction[correct_next_token])

Model B is valid: True
Model B target probability: 0.01


## The Top Token Can Be Wrong

A model can rank `"rug"` first while still assigning meaningful probability to the observed target `"mat"`.

Top-one correctness discards information that the target probability retains.


In [11]:
wrong_top_prediction = {
    "mat": 0.35,
    "rug": 0.40,
    "yard": 0.05,
    "food": 0.03,
    "dog": 0.04,
    "cat": 0.03,
    "bird": 0.03,
    "tree": 0.02,
    ".": 0.05,
}

print_prediction_report(
    "Wrong-top-token model",
    input_context,
    correct_next_token,
    wrong_top_prediction,
    vocabulary,
)

Model: Wrong-top-token model
Input context: 'the cat sat on the'
Correct next token: 'mat'
Target probability: 0.35
Top predicted token: 'rug'
Top probability: 0.4


This prediction is worse than Model A for the observed target but much better than Model B's target probability of `0.01`.


## Sampling Can Differ From the Target

If a model assigns the target probability `0.60`, sampling still selects another token with total probability `0.40`.

A non-target sample does not retroactively change the quality of the distribution that was evaluated.

Generation quality and prediction evaluation therefore should not be conflated.


## Define Multiple Evaluation Examples

Real model comparison aggregates evidence across many held-out context-target pairs.


In [12]:
from typing import TypedDict  # noqa: I001


class PredictionExample(TypedDict):
    input_context: str
    correct_next_token: str
    model_a_prediction: dict[str, float]
    model_b_prediction: dict[str, float]


examples: list[PredictionExample] = [
    {
        "input_context": "the cat sat on the",
        "correct_next_token": "mat",
        "model_a_prediction": model_a_prediction,
        "model_b_prediction": model_b_prediction,
    },
    {
        "input_context": "the dog ran in the",
        "correct_next_token": "yard",
        "model_a_prediction": {
            "mat": 0.05,
            "rug": 0.05,
            "yard": 0.55,
            "food": 0.05,
            "dog": 0.05,
            "cat": 0.05,
            "bird": 0.05,
            "tree": 0.05,
            ".": 0.10,
        },
        "model_b_prediction": {
            "mat": 0.25,
            "rug": 0.20,
            "yard": 0.05,
            "food": 0.15,
            "dog": 0.10,
            "cat": 0.10,
            "bird": 0.05,
            "tree": 0.05,
            ".": 0.05,
        },
    },
    {
        "input_context": "the bird sat in the",
        "correct_next_token": "tree",
        "model_a_prediction": {
            "mat": 0.05,
            "rug": 0.05,
            "yard": 0.10,
            "food": 0.05,
            "dog": 0.05,
            "cat": 0.05,
            "bird": 0.05,
            "tree": 0.50,
            ".": 0.10,
        },
        "model_b_prediction": {
            "mat": 0.20,
            "rug": 0.20,
            "yard": 0.20,
            "food": 0.10,
            "dog": 0.10,
            "cat": 0.10,
            "bird": 0.05,
            "tree": 0.01,
            ".": 0.04,
        },
    },
]

for example in examples:
    assert_valid_prediction(example["model_a_prediction"], vocabulary)
    assert_valid_prediction(example["model_b_prediction"], vocabulary)

print("Validated examples:", len(examples))

Validated examples: 3


## Print a Compact Target-Probability Table

Each row extracts exactly one probability from each model distribution using the observed target.


In [13]:
print("Example | Context                  | Target | Model A | Model B")
print("-" * 68)
for example_number, example in enumerate(examples):
    target = example["correct_next_token"]
    model_a_probability = target_probability(
        example["model_a_prediction"], target, vocabulary
    )
    model_b_probability = target_probability(
        example["model_b_prediction"], target, vocabulary
    )
    print(
        f"{example_number:>7} | {example['input_context']:<24} | "
        f"{target:>6} | {model_a_probability:>7.3f} | "
        f"{model_b_probability:>7.3f}"
    )

Example | Context                  | Target | Model A | Model B
--------------------------------------------------------------------
      0 | the cat sat on the       |    mat |   0.600 |   0.010
      1 | the dog ran in the       |   yard |   0.550 |   0.050
      2 | the bird sat in the      |   tree |   0.500 |   0.010


## Average Target Probability for Intuition

The arithmetic mean gives an intuitive summary here, but it is not the standard final language-model scoring rule.

Chapter 21 will use negative log loss, which penalizes very low target probabilities much more strongly and has better statistical properties.


In [14]:
def average_target_probability(
    examples: list[PredictionExample],
    prediction_key: str,
    vocabulary: list[str],
) -> float:
    if prediction_key not in {"model_a_prediction", "model_b_prediction"}:
        raise ValueError("Unknown prediction key.")

    probabilities = []
    for example in examples:
        prediction = (
            example["model_a_prediction"]
            if prediction_key == "model_a_prediction"
            else example["model_b_prediction"]
        )
        probabilities.append(
            target_probability(
                prediction,
                example["correct_next_token"],
                vocabulary,
            )
        )
    return sum(probabilities) / len(probabilities)


model_a_average = average_target_probability(
    examples,
    "model_a_prediction",
    vocabulary,
)
model_b_average = average_target_probability(
    examples,
    "model_b_prediction",
    vocabulary,
)

print("Model A average target probability:", model_a_average)
print("Model B average target probability:", model_b_average)
assert model_a_average > model_b_average

Model A average target probability: 0.5499999999999999
Model B average target probability: 0.023333333333333334


## Reject Vocabulary Mismatches

A real fixed-vocabulary model emits one probability for every output token, so a missing target entry is a malformed prediction rather than merely a low score.


In [15]:
incomplete_prediction = {
    "rug": 0.50,
    "yard": 0.25,
    ".": 0.25,
}

try:
    assert_valid_prediction(incomplete_prediction, vocabulary)
except ValueError as error:
    print("Caught expected error:", error)

Caught expected error: Prediction keys do not match the vocabulary. Missing: ['bird', 'cat', 'dog', 'food', 'mat', 'tree']; extra: [].


## Zero Target Probability Is a Severe Failure

A valid distribution can assign the observed target no probability at all.

The next chapter's negative log loss will make the consequence of this zero explicit.


In [16]:
zero_target_prediction = {
    "mat": 0.00,
    "rug": 0.50,
    "yard": 0.20,
    "food": 0.10,
    "dog": 0.05,
    "cat": 0.05,
    "bird": 0.03,
    "tree": 0.02,
    ".": 0.05,
}

assert_valid_prediction(zero_target_prediction, vocabulary)
print(
    "Target probability:",
    target_probability(
        zero_target_prediction,
        "mat",
        vocabulary,
    ),
)

Target probability: 0.0


## Prediction Quality Is a Narrow Claim

High target probability means the model predicted one observed token well under the chosen scoring idea.

It does not by itself demonstrate semantic understanding, causal reasoning, calibration, robustness, or useful generation.

Those broader claims require additional evidence and different evaluations.


## Complete One-Example Pipeline

The final cell validates both distributions, extracts their target probabilities, and reports the better prediction.


In [17]:
final_context = "the cat sat on the"
final_target = "mat"

assert_valid_prediction(model_a_prediction, vocabulary)
assert_valid_prediction(model_b_prediction, vocabulary)

final_a_probability = target_probability(
    model_a_prediction,
    final_target,
    vocabulary,
)
final_b_probability = target_probability(
    model_b_prediction,
    final_target,
    vocabulary,
)
final_better_model = better_prediction_for_target(
    "Model A",
    model_a_prediction,
    "Model B",
    model_b_prediction,
    final_target,
    vocabulary,
)

print("Input context:", repr(final_context))
print("Correct next token:", repr(final_target))
print("Model A target probability:", final_a_probability)
print("Model B target probability:", final_b_probability)
print("Better prediction:", final_better_model)

assert final_better_model == "Model A"

Input context: 'the cat sat on the'
Correct next token: 'mat'
Model A target probability: 0.6
Model B target probability: 0.01
Better prediction: Model A


## Common Mistakes

- Evaluate the probability assigned to the observed target rather than only top-one accuracy.

- Validate that every model predicts over the same complete vocabulary.

- Do not require probability one for every correct token in uncertain language.

- Do not use one lucky observed outcome to justify extreme confidence.

- Keep the full prediction distribution distinct from one sampled generation token.

- Aggregate many held-out examples before drawing conclusions about a model.

- Treat high target probability as prediction evidence rather than automatic proof of understanding.


## Takeaways

For one held-out context-target pair, a better prediction assigns more probability to the token that actually came next.

The target need not be the top-ranked token for its probability to carry useful evaluation information.

Language admits multiple plausible continuations, so uncertainty can be appropriate even when only one continuation appears in a particular example.

Single-example target probability is the foundation, but reliable model evaluation aggregates this idea across many examples.

The next chapter converts target probability into negative log loss, where higher target probability produces lower loss and zero target probability creates an extreme penalty.
